In [9]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Описание данных

**Целевая переменная:** `SeriousDlqin2yrs` — был ли у клиента просрочка 90+ дней или хуже за последние 2 года.

| Признак | Описание | Тип данных |
|---------|----------|-------------|
| `SeriousDlqin2yrs` | Просрочка 90+ дней или хуже за последние 2 года | Y/N (целевой) |
| `RevolvingUtilizationOfUnsecuredLines` | Отношение общего остатка по кредитным картам и персональным кредитным линиям (кроме недвижимости и рассрочки) к сумме кредитных лимитов | доля |
| `age` | Возраст заёмщика в годах | целое число |
| `NumberOfTime30-59DaysPastDueNotWorse` | Количество раз, когда заёмщик допускал просрочку 30–59 дней (но не более) за последние 2 года | целое число |
| `DebtRatio` | Отношение ежемесячных выплат по долгам, алиментов, расходов на жизнь к валовому месячному доходу | доля |
| `MonthlyIncome` | Ежемесячный доход | вещественное число |
| `NumberOfOpenCreditLinesAndLoans` | Количество открытых кредитов (например, автокредит, ипотека) и кредитных линий (например, кредитные карты) | целое число |
| `NumberOfTimes90DaysLate` | Количество раз, когда заёмщик допускал просрочку 90+ дней | целое число |
| `NumberRealEstateLoansOrLines` | Количество ипотечных кредитов и кредитов под недвижимость, включая возобновляемые кредитные линии под залог недвижимости | целое число |
| `NumberOfTime60-89DaysPastDueNotWorse` | Количество раз, когда заёмщик допускал просрочку 60–89 дней (но не более) за последние 2 года | целое число |
| `NumberOfDependents` | Количество иждивенцев в семье (не включая самого заёмщика) — супруг(а), дети и т.д. | целое число |


In [10]:
train_file = Path('data.csv')
if not train_file.exists():
    train_file = Path('hw1/data.csv')

In [11]:
df = pd.read_csv(train_file)
df.head()

,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,SeriousDlqin2yrs
0,0.114987,62,0,1841.000000,NaN,5,0,1,0,2.0,0
1,0.008705,73,0,0.498553,3800.0,6,0,1,0,0.0,0
2,0.214501,32,0,0.211999,3716.0,8,0,0,0,2.0,0
3,1.000000,60,0,118.000000,NaN,5,0,0,0,0.0,0
4,0.230493,60,0,1.017328,3000.0,10,0,1,0,0.0,0


In [12]:
X = df.drop('SeriousDlqin2yrs', axis=1)
y = df['SeriousDlqin2yrs']

In [13]:
feature_names = X.columns.tolist()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42)),
])

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. I

## Задание

Обучить линейную модель предсказания целевой переменной. Оценить качество. Считается, что дать кредит человеку, который допустит просрочку, для банка в 5 раз дороже, чем не дать платёжеспособному.

Ноутбук должен быть рабочим. Также должна быть реализована функция predict(model, data_path), возвращающая предсказанные значения. 

In [14]:
def predict(model, data_path):
    df = pd.read_csv(data_path)
    X_input = df.drop(columns=['SeriousDlqin2yrs'], errors='ignore')
    X_input = X_input.reindex(columns=model.feature_names_)

    proba = model.predict_proba(X_input)[:, 1]
    threshold = getattr(model, 'threshold_', 0.5)
    return (proba >= threshold).astype(int)


In [15]:
def business_cost(y_true, y_pred, fn_cost=5, fp_cost=1):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    total_cost = fn_cost * fn + fp_cost * fp
    avg_cost = total_cost / len(y_true)
    return total_cost, avg_cost, (tn, fp, fn, tp)

valid_proba = model.predict_proba(X_valid)[:, 1]
threshold_grid = np.linspace(0.01, 0.99, 199)
threshold_rows = []

for threshold in threshold_grid:
    valid_pred = (valid_proba >= threshold).astype(int)
    total_cost, avg_cost, _ = business_cost(y_valid, valid_pred)
    threshold_rows.append((threshold, total_cost, avg_cost))

threshold_df = pd.DataFrame(threshold_rows, columns=['Порог', 'Суммарная стоимость', 'Средняя стоимость'])
best_threshold = threshold_df.sort_values(['Суммарная стоимость', 'Порог'], ascending=[True, True]).iloc[0]['Порог']

model.feature_names_ = feature_names
model.threshold_ = float(best_threshold)

# Accuracy здесь недостаточна: классы несбалансированы, и цена ошибок разная.
# Поэтому подбираем порог по функции стоимости 5 * FN + FP, а не фиксируем 0.5.
threshold_df.nsmallest(5, 'Суммарная стоимость')

,Порог,Суммарная стоимость,Средняя стоимость
25,0.133737,6596,0.274833
24,0.128788,6604,0.275167
23,0.123838,6621,0.275875
26,0.138687,6632,0.276333
27,0.143636,6662,0.277583


In [16]:
import tempfile

test_proba = model.predict_proba(X_test)[:, 1]
test_pred_default = (test_proba >= 0.5).astype(int)
test_pred_best = (test_proba >= model.threshold_).astype(int)

default_total_cost, default_avg_cost, default_cm = business_cost(y_test, test_pred_default)
best_total_cost, best_avg_cost, best_cm = business_cost(y_test, test_pred_best)

print(f'Выбранный порог по валидации: {model.threshold_:.3f}')
print(f'Accuracy при пороге 0.5: {accuracy_score(y_test, test_pred_default):.4f}')
print(f'Accuracy при лучшем пороге: {accuracy_score(y_test, test_pred_best):.4f}')
print(f'Precision при лучшем пороге: {precision_score(y_test, test_pred_best, zero_division=0):.4f}')
print(f'Recall при лучшем пороге: {recall_score(y_test, test_pred_best, zero_division=0):.4f}')
print(f'ROC-AUC: {roc_auc_score(y_test, test_proba):.4f}')
print(f'При пороге 0.5 -> FP: {default_cm[1]}, FN: {default_cm[2]}, суммарная стоимость: {default_total_cost}')
print(f'При лучшем пороге -> FP: {best_cm[1]}, FN: {best_cm[2]}, суммарная стоимость: {best_total_cost}')
print(f'Средняя стоимость ошибки при лучшем пороге: {best_avg_cost:.4f}')
print('Матрица ошибок при лучшем пороге [TN, FP; FN, TP]:')
print(np.array([[best_cm[0], best_cm[1]], [best_cm[2], best_cm[3]]]))

train_preds = predict(model, train_file)
print(f'Предсказаний для обучающего файла: {len(train_preds)} строк')

with tempfile.NamedTemporaryFile(suffix='.csv', delete=False) as tmp:
    predict_check_path = Path(tmp.name)

df.drop(columns=['SeriousDlqin2yrs']).to_csv(predict_check_path, index=False)
predict_without_target = predict(model, predict_check_path)
predict_check_path.unlink(missing_ok=True)

print(f'Предсказаний для файла без целевого столбца: {len(predict_without_target)} строк')
print(f'Проверка длины пройдена: {len(predict_without_target) == len(df)}')

Выбранный порог по валидации: 0.134
Accuracy при пороге 0.5: 0.9343
Accuracy при лучшем пороге: 0.9127
Precision при лучшем пороге: 0.3232
Recall при лучшем пороге: 0.2793
ROC-AUC: 0.6833
При пороге 0.5 -> FP: 44, FN: 1532, суммарная стоимость: 7704
При лучшем пороге -> FP: 938, FN: 1156, суммарная стоимость: 6718
Средняя стоимость ошибки при лучшем пороге: 0.2799
Матрица ошибок при лучшем пороге [TN, FP; FN, TP]:
[[21458   938]
 [ 1156   448]]
Предсказаний для обучающего файла: 120000 строк
Предсказаний для файла без целевого столбца: 120000 строк
Проверка длины пройдена: True
